# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. The sample is two NIH awards and 438 expenditure rows, only 4 of which are flagged ineligible. Five-fold stratified cross-validation with 4 positives is below what the splitter asks for, so scikit-learn warns, and the AUC from 4 positive rows is close to meaningless.

The number to cite is the one measured on the full dataset: detection AUC 0.8633 (logistic regression) and 0.8826 (gradient boosting), 5-fold CV on 16,356 expenditure rows, prevalence 0.021.


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
import numpy as np
flat = pd.read_parquet(SAMPLE / "d09_sample.parquet")
# The script expects the full dataset's three tables. The sample ships one denormalised table (expenditure joined
# with award attributes), so split it back into the three tables with the same column names. No values are changed.
(work / "DATASET").mkdir()
flat[["expenditure_id", "award_id", "date", "category", "amount", "description", "eligible", "ineligibility_reason"]] \
    .to_parquet(work / "DATASET" / "expenditure.parquet", index=False)
flat.drop_duplicates("award_id")[["award_id", "funder", "program", "awarded_amount", "start_date", "end_date",
                                  "pi_id", "department", "funder_policy_id", "currency"]] \
    .to_parquet(work / "DATASET" / "award.parquet", index=False)
flat.drop_duplicates("award_id")[["award_id", "annual_direct", "n_fte", "negotiated_fa_rate", "overhead_rate"]] \
    .to_parquet(work / "DATASET" / "award_financials.parquet", index=False)
shutil.copy(SCRIPT, work / "benchmark_script.py")
r = subprocess.run([sys.executable, "benchmark_script.py", "--dataset", str(work / "DATASET"),
                    "--out", str(work / "benchmark_results.parquet")], cwd=work, capture_output=True, text=True)
print(clean(r.stdout))
print(clean(r.stderr[-1500:]))
r.check_returncode()
results = pd.read_parquet(work / "benchmark_results.parquet")
results

              model    auc   n  prevalence
logistic_regression 0.8329 438      0.0091
  gradient_boosting 0.4263 438      0.0091

/usr/local/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



,model,auc,n,prevalence
0,logistic_regression,0.8329,438,0.0091
1,gradient_boosting,0.4263,438,0.0091


## Reading the output

Read the AUC column with the sample size in mind: both models are fitted on roughly 350 rows with only four positives, and the sample covers a single funder, so the funder and category dummies carry almost no information. In the executed run logistic regression reaches AUC 0.833 and gradient boosting 0.426, which is below chance: with four positive rows the result swings from one extreme to the other. Only the full-dataset figure above is a benchmark.